# 05. SHAP 심층 분석

**목표**: 모델이 독성을 예측하는 근거를 화학적으로 해석

1. LogP Dependence Plot → 임계값 찾기
2. 고독성 vs 저독성 분자 구조 시각화
3. fp_2047 서브구조 역추적


## 0. 라이브러리 + 데이터 + 모델 로드

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import shap
import warnings
warnings.filterwarnings('ignore')

from rdkit import Chem
from rdkit.Chem import AllChem, Draw, rdMolDescriptors, Descriptors
from rdkit.Chem.Draw import rdMolDraw2D
from IPython.display import display, Image
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# 데이터 로드
df = pd.read_csv('../data/processed/chembl_features.csv')
print(f'데이터 로드: {df.shape}')

# 피처 / 타겟 분리
descriptor_cols = [
    'MolWt','LogP','HBD','HBA','TPSA',
    'RingCount','RotBonds','AromaticRings',
    'HeavyAtoms','FractionCSP3'
]
morgan_cols = [f'fp_{i}' for i in range(2048)]
feature_names = descriptor_cols + morgan_cols

X = df[feature_names].values
y = df['label'].values

X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y, df.index,
    test_size=0.2, random_state=42, stratify=y
)

df_test = df.loc[idx_test].reset_index(drop=True)

# Random Forest 학습
rf = RandomForestClassifier(
    n_estimators=100, random_state=42,
    class_weight='balanced', n_jobs=-1
)
rf.fit(X_train, y_train)
auc = roc_auc_score(y_test, rf.predict_proba(X_test)[:,1])
print(f'RF AUC: {auc:.4f}')

# SHAP 계산
explainer = shap.TreeExplainer(rf)
sv = explainer.shap_values(X_test)
sv_positive = sv[1]  # 고독성 클래스
print(f'SHAP values shape: {sv_positive.shape}')

## 1. LogP Dependence Plot — 임계값 찾기

In [ ]:
logp_idx  = feature_names.index('LogP')
logp_vals = X_test[:, logp_idx]
shap_logp = sv_positive[:, logp_idx]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 왼쪽: Scatter (LogP vs SHAP, 실제 라벨 색상)
sc = axes[0].scatter(
    logp_vals, shap_logp,
    c=y_test, cmap='RdBu_r', alpha=0.5, edgecolors='none'
)
axes[0].axhline(0, color='gray', linestyle='--', linewidth=1)
axes[0].axvline(0, color='red',  linestyle='--', linewidth=1, label='LogP=0')
axes[0].set_xlabel('LogP 값', fontsize=12)
axes[0].set_ylabel('SHAP value (독성 기여도)', fontsize=12)
axes[0].set_title('LogP vs SHAP value\n(빨강=고독성, 파랑=저독성)', fontsize=12)
plt.colorbar(sc, ax=axes[0], label='실제 라벨')
axes[0].legend()

# 오른쪽: LogP 구간별 고독성 비율
bins = [-5, -2, 0, 2, 4, 6, 8, 15]
labels = ['<-2','-2~0','0~2','2~4','4~6','6~8','>8']
logp_bin = pd.cut(logp_vals, bins=bins, labels=labels)
bin_df = pd.DataFrame({
    'logp_bin': logp_bin,
    'label': y_test,
    'shap': shap_logp
})
tox_rate = bin_df.groupby('logp_bin', observed=True)['label'].mean()
count    = bin_df.groupby('logp_bin', observed=True)['label'].count()

bars = axes[1].bar(tox_rate.index, tox_rate.values,
                   color='salmon', edgecolor='white')
axes[1].set_xlabel('LogP 구간', fontsize=12)
axes[1].set_ylabel('고독성 비율', fontsize=12)
axes[1].set_title('LogP 구간별 고독성 비율', fontsize=12)
axes[1].axhline(y_test.mean(), color='blue',
                linestyle='--', label=f'전체 평균 {y_test.mean():.1%}')
for bar, c in zip(bars, count):
    axes[1].text(bar.get_x()+bar.get_width()/2,
                 bar.get_height()+0.01,
                 f'n={c}', ha='center', fontsize=9)
axes[1].legend()

plt.tight_layout()
plt.savefig('../data/processed/shap_logp_detail.png', dpi=150, bbox_inches='tight')
plt.show()

# 임계값 분석
print('=== LogP 임계값 분석 ===')
print(f'SHAP>0 화합물(고독성 기여) 평균 LogP: {logp_vals[shap_logp>0].mean():.2f}')
print(f'SHAP<0 화합물(저독성 기여) 평균 LogP: {logp_vals[shap_logp<0].mean():.2f}')
print(f'\nLogP > 2: 고독성 비율 = {bin_df[logp_vals>2]["label"].mean():.1%}')
print(f'LogP < 0: 고독성 비율 = {bin_df[logp_vals<0]["label"].mean():.1%}')

## 2. 고독성 vs 저독성 분자 구조 시각화

In [ ]:
# 예측 확률 추가
df_test = df_test.copy()
df_test['pred_proba'] = rf.predict_proba(X_test)[:, 1]
df_test['true_label'] = y_test

# 고독성 상위 8개 (실제 고독성이면서 예측 확률 높은 것)
high_tox = (
    df_test[df_test['true_label']==1]
    .nlargest(8, 'pred_proba')['canonical_smiles']
    .tolist()
)

# 저독성 상위 8개 (실제 저독성이면서 예측 확률 낮은 것)
low_tox = (
    df_test[df_test['true_label']==0]
    .nsmallest(8, 'pred_proba')['canonical_smiles']
    .tolist()
)

mols_high = [Chem.MolFromSmiles(s) for s in high_tox if Chem.MolFromSmiles(s)]
mols_low  = [Chem.MolFromSmiles(s) for s in low_tox  if Chem.MolFromSmiles(s)]

# LogP 값 뽑기
logp_high = [round(Descriptors.MolLogP(m), 2) for m in mols_high]
logp_low  = [round(Descriptors.MolLogP(m), 2) for m in mols_low]

# 시각화
img_high = Draw.MolsToGridImage(
    mols_high, molsPerRow=4, subImgSize=(350, 250),
    legends=[f'고독성 | LogP={lp}' for lp in logp_high]
)
img_low = Draw.MolsToGridImage(
    mols_low, molsPerRow=4, subImgSize=(350, 250),
    legends=[f'저독성 | LogP={lp}' for lp in logp_low]
)

img_high.save('../data/processed/mol_high_tox.png')
img_low.save('../data/processed/mol_low_tox.png')

print('=== 고독성 화합물 (예측 확률 상위 8개) ===')
print(f'평균 LogP: {np.mean(logp_high):.2f}')
display(img_high)

print('\n=== 저독성 화합물 (예측 확률 하위 8개) ===')
print(f'평균 LogP: {np.mean(logp_low):.2f}')
display(img_low)

## 3. fp_2047 서브구조 역추적

SHAP 1위 피처 fp_2047이 어떤 화학 서브구조에 해당하는지 추적

In [ ]:
# fp_2047 = Morgan FP의 2047번째 비트
# 이 비트가 켜진 화합물에서 어떤 원자 환경이 이 비트를 만드는지 역추적

fp_bit = 2047
fp_col_idx = len(descriptor_cols) + fp_bit  # 전체 피처에서의 인덱스

# 테스트셋에서 fp_2047=1 vs 0 화합물 독성 비교
bit_vals = X_test[:, fp_col_idx]
df_bit = pd.DataFrame({
    'bit': bit_vals,
    'label': y_test,
    'smiles': df_test['canonical_smiles'].values
})

n_active   = (bit_vals == 1).sum()
n_inactive = (bit_vals == 0).sum()
tox_active   = df_bit[df_bit['bit']==1]['label'].mean()
tox_inactive = df_bit[df_bit['bit']==0]['label'].mean()

print('=== fp_2047 활성화 여부별 독성 비율 ===')
print(f'fp_2047=1 (활성): {n_active}개, 고독성 비율 = {tox_active:.1%}')
print(f'fp_2047=0 (비활): {n_inactive}개, 고독성 비율 = {tox_inactive:.1%}')
print(f'→ 독성 비율 차이: {tox_active - tox_inactive:+.1%}')

In [ ]:
# fp_2047=1인 화합물에서 실제로 어떤 서브구조가 이 비트를 켜는지 시각화

# fp_2047 활성화 화합물 추출
active_smiles = df_bit[df_bit['bit']==1]['smiles'].tolist()[:6]

highlight_mols = []
highlight_atoms_list = []

for smi in active_smiles:
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        continue
    
    # Morgan FP에서 각 비트가 어떤 원자에서 유래하는지 추적
    bi = {}  # bit_info
    fp = AllChem.GetMorganFingerprintAsBitVect(
        mol, radius=2, nBits=2048, bitInfo=bi
    )
    
    if fp_bit in bi:
        # 이 비트를 만드는 원자들
        highlight_atoms = list(set([atom_idx for atom_idx, _ in bi[fp_bit]]))
        highlight_mols.append(mol)
        highlight_atoms_list.append(highlight_atoms)

print(f'fp_2047 활성 + 시각화 가능 화합물: {len(highlight_mols)}개')

if highlight_mols:
    img = Draw.MolsToGridImage(
        highlight_mols[:6],
        molsPerRow=3,
        subImgSize=(400, 300),
        highlightAtomLists=highlight_atoms_list[:6],
        legends=[f'fp_2047 활성 화합물 {i+1}' for i in range(len(highlight_mols[:6]))]
    )
    img.save('../data/processed/fp2047_substructure.png')
    display(img)
    print('→ 하이라이트된 원자가 fp_2047을 만드는 서브구조')
else:
    print('bitInfo 역추적 실패 — 아래 대안 실행')

In [ ]:
# fp_2047 활성 화합물의 화학적 특성 비교
# → 어떤 분자 특성을 가진 화합물이 이 비트를 켜는가?

def get_props(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None
    return {
        'MolWt': Descriptors.MolWt(mol),
        'LogP':  Descriptors.MolLogP(mol),
        'HBD':   rdMolDescriptors.CalcNumHBD(mol),
        'HBA':   rdMolDescriptors.CalcNumHBA(mol),
        'Rings': rdMolDescriptors.CalcNumRings(mol),
        'AromaticRings': rdMolDescriptors.CalcNumAromaticRings(mol),
    }

props_active   = pd.DataFrame([get_props(s) for s in df_bit[df_bit['bit']==1]['smiles']]).dropna()
props_inactive = pd.DataFrame([get_props(s) for s in df_bit[df_bit['bit']==0]['smiles']]).dropna()

print('=== fp_2047 활성 vs 비활성 화합물 특성 비교 ===')
comparison = pd.DataFrame({
    'fp_2047=1 (활성)': props_active.mean().round(2),
    'fp_2047=0 (비활)': props_inactive.mean().round(2),
})
comparison['차이'] = (comparison['fp_2047=1 (활성)'] - comparison['fp_2047=0 (비활)']).round(2)
print(comparison)

print('\n=== 해석 ===')
for prop in ['LogP', 'MolWt', 'Rings', 'AromaticRings']:
    diff = comparison.loc[prop, '차이']
    direction = '높음' if diff > 0 else '낮음'
    print(f'  {prop}: 활성 화합물이 평균 {abs(diff)} {direction}')

## 4. 종합 인사이트 정리

In [ ]:
print('=' * 60)
print('SHAP 심층 분석 종합 인사이트')
print('=' * 60)

print('''
[1] LogP와 세포독성
  - LogP > 2 이상인 화합물에서 고독성 비율 급증
  - 지용성이 높을수록 세포막 투과 → 세포내 축적 → 독성 증가
  - ADC 페이로드 설계 시 LogP 조절이 핵심 파라미터

[2] Morgan FP 서브구조 (fp_2047)
  - SHAP 1위 피처: 특정 분자 서브구조 패턴
  - 이 서브구조를 가진 화합물의 고독성 비율이 유의미하게 높음
  - 구조적 특성: 방향족 링 + 높은 LogP 경향

[3] 분자량(MolWt)의 낮은 기여
  - MolWt SHAP ≈ 0 → 크기보다 구조/극성이 독성 결정
  - ADC 페이로드 최적화 시 분자량보다 LogP/극성 조절 우선

[포트폴리오 스토리]
  "SHAP 분석을 통해 LogP가 임계값(~2)을 넘으면
   세포독성이 급증하는 패턴을 발견.
   이는 ADC 페이로드 후보 스크리닝 시
   LogP < 2 필터를 적용하는 실용적 기준을 제시함"
''')

print('=' * 60)
print('저장된 파일:')
import os
for f in os.listdir('../data/processed'):
    if f.endswith('.png') or f.endswith('.csv'):
        print(f'  data/processed/{f}')